## AI MCP Client – Agents SDK

Dieses Notebook verbindet ein LLM über das OpenAI Agents SDK mit dem unveränderten Filesystem-MCP-Server aus Notebook 10.

Die sechs MCP-Tools `pwd`, `ls`, `cat`, `grep`, `df` und `uname` werden automatisch bereitgestellt. Das LLM wählt anhand der jeweiligen Frage selbst das passende Tool und dessen Argumente aus. Das Agents SDK führt die MCP-Aufrufe aus und übergibt die Ergebnisse an das LLM. Der Client benötigt keinen eigenen Tool-Loop.

Die LLM-Konfiguration stammt aus `~/data/env-ollama.py`. Voraussetzung ist ein laufender MCP-Server unter `http://localhost:8000/mcp` und ein Modell mit Tool-Calling-Unterstützung.

### 1. Umgebung einrichten

In [ ]:
%run ~/data/env-ollama.py

from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel, set_tracing_disabled
from agents.mcp import MCPServerStreamableHttp
from pprint import pprint

MCP_URL = "http://localhost:8000/mcp"

# Für lokale Ollama-Modelle kein OpenAI-Tracing senden.
set_tracing_disabled(True)

llm = AsyncOpenAI(
    api_key=OPENAI_API_KEY,
    base_url=AI_BASE_URL,
)

### 2. Agenten definieren

Die Funktion `frage()` führt eine frei formulierte Anfrage über den Agenten aus. Das Agents SDK übernimmt die Tool-Auswahl, die MCP-Aufrufe und die Weiterverarbeitung der Resultate. Zusätzlich werden die SDK-Ereignisse angezeigt, damit echte MCP-Aufrufe überprüfbar sind.

In [ ]:
async def frage(anfrage: str):
    async with MCPServerStreamableHttp(
        name="Filesystem",
        params={"url": MCP_URL},
        cache_tools_list=True,
    ) as server:
        agent = Agent(
            name="Filesystem-Assistent",
            instructions=(
                "Du bist ein Assistent für Dateisystem- und Systemabfragen. "
                "Nutze für jede Anfrage die passenden MCP-Tools des verbundenen Servers. "
                "Wähle Tool und Argumente anhand der Benutzerfrage selbst aus. "
                "Erfinde keine Ergebnisse und simuliere keine Tool-Aufrufe. "
                "Antworte kurz auf Deutsch."
            ),
            model=OpenAIChatCompletionsModel(
                model=AI_MODEL,
                openai_client=llm,
            ),
            mcp_servers=[server],
        )

        # Das Agents SDK steuert die MCP-Aufrufe selbstständig.
        result = await Runner.run(agent, anfrage, max_turns=10)

    #print("\n===== SDK-Ereignisse =====")
    #pprint(result.new_items)

    # Allgemeine Erklärungen ohne tatsächlichen Tool-Aufruf nicht akzeptieren.
    if not any(item.type == "tool_call_item" for item in result.new_items):
        raise RuntimeError("Das LLM hat kein MCP-Tool aufgerufen. Prüfe AI_MODEL und Tool-Calling-Unterstützung.")

    print("\n===== Antwort =====")
    print(result.final_output)
    return result

### 3. Fragen an den MCP-Agenten

Jede der folgenden Zellen enthält eine eigene Benutzerfrage. Die Zellen können unabhängig voneinander ausgeführt werden. Die Toolnamen werden dem Agenten **nicht** vorgegeben: Er entscheidet anhand des Textes.

#### Aktuelles Arbeitsverzeichnis

In [ ]:
result = await frage("In welchem Verzeichnis befindet sich der Filesystem-MCP-Server?")

#### Dateien und Verzeichnisse anzeigen

In [ ]:
result = await frage("Zeige mir die Dateien und Unterverzeichnisse in meinem Home-Verzeichnis.")

#### Datei lesen

In [ ]:
result = await frage("Zeige mir den Inhalt der Datei .profile in meinem Home-Verzeichnis.")

#### Text in einer Datei suchen

In [ ]:
result = await frage("Suche in der Datei .profile nach dem Text PATH und zeige maximal 10 Treffer.")

#### Freien Speicherplatz anzeigen

In [ ]:
result = await frage("Wie viel Speicherplatz ist auf den Dateisystemen noch frei?")

#### Betriebssystem- und Kernel-Informationen

In [ ]:
result = await frage("Welcher Kernel und welche Rechnerarchitektur werden verwendet?")

### 4. Eigene Frage

Den Text in der nächsten Zelle direkt ändern und die Zelle erneut ausführen.

In [ ]:
result = await frage("Welche Dateien liegen im Verzeichnis data?")

### Hinweise

- Für die Datei-Beispiele muss `.profile` im Home-Verzeichnis vorhanden sein; andernfalls den Dateinamen in den beiden betreffenden Fragen ersetzen.
- Die Ausgabe **SDK-Ereignisse** macht ersichtlich, ob ein MCP-Tool tatsächlich ausgeführt wurde.
- Ein Modell ohne Tool-Calling-Unterstützung eignet sich nicht für diese Übung, z.B. `gemma3:4b`.
- Ein fehlender Tool-Aufruf führt hier zu einer Fehlermeldung statt zu einem scheinbar erfolgreichen Resultat.